In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

# 1. 환경(패키지 및 환경변수)
- pip install openai langchain-upstage

In [2]:
%pip install langchain-upstage

   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------  2.4/2.4 MB 12.2 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 10.4 MB/s eta 0:00:00

   ---------------------------------------- 0/3 [pypdf]
   ---------------------------------------- 0/3 [pypdf]
   ---------------------------------------- 0/3 [pypdf]
  Attempting uninstall: tokenizers
   ---------------------------------------- 0/3 [pypdf]
    Found existing installation: tokenizers 0.21.2
   ---------------------------------------- 0/3 [pypdf]
    Uninstalling tokenizers-0.21.2:
   ---------------------------------------- 0/3 [pypdf]
      Successfully uninstalled tokenizers-0.21.2
   ---------------------------------------- 0/3 [pypdf]
   ------------- -------------------------- 1/3 [tokenizers]
   ------------- -------------------------- 1/3 [tokenizers]
   ---------------------------------------- 3/3 [langchain-upstage]

Note: you may need to 

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.53.3 requires tokenizers<0.22,>=0.21, but you have tokenizers 0.20.3 which is incompatible.


In [3]:
from dotenv import load_dotenv
import os
load_dotenv()
UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")

# 2. LLM 답변 생성
## 2.1 OpenAI SDK를 사용

In [4]:
from openai import OpenAI # openai==1.52.2
 
client = OpenAI(
    api_key=UPSTAGE_API_KEY,
    base_url="https://api.upstage.ai/v1"
)
 
stream = client.chat.completions.create(
    model="solar-pro2",
    messages=[
        {
            "role": "user",
            "content": "2020년 월드시리즈 누가 우승했나요?"
        }
    ],
    stream=False,
)
 
# for chunk in stream:
#     if chunk.choices[0].delta.content is not None:
#         print(chunk.choices[0].delta.content, end="")

In [7]:
print(stream.choices[0].message.content)

2020년 월드시리즈에서는 **로스앤젤레스 다저스**가 우승했습니다.  

다저스는 탬파베이 레이스를 **4승 2패**로 꺾고, 32년 만에 통산 28번째 월드 시리즈 타이틀을 획득했습니다. 특히 코로나19 팬데믹으로 무관중 경기로 진행된 시리즈에서도 강력한 투수진과 타격으로 압도적인 모습을 보였습니다.  

주요 MVP는 다저스의 코리 시거(Corey Seager)가 수상했으며, 그는 시리즈 동안 0.379 타율과 2홈런, 6타점을 기록하며 팀의 승리를 이끌었습니다.


## 2.2 LangChain을 사용
- 발급받은 API KEY를 UPSTAGE_API_KEY라고 저장하면 별도의 설정없이 ChatUpstage 바로 사용

In [8]:
from langchain_upstage import ChatUpstage
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatUpstage(
    model = "solar-pro2",
)

message = [
    SystemMessage(content = "당신은 친절하게 답변하는 비서입니다."),
    HumanMessage(content = "2020년 월드시리즈 누가 우승했나요?")
]

ai_message = llm.invoke(message)
ai_message

AIMessage(content='2020년 월드 시리즈에서는 **로스앤젤레스 다저스**가 우승했습니다!  \nLA 다저스는 탬파베이 레이스와 맞붙어 6차전까지 가는 접전 끝에 4승 2패로 우승을 차지했으며, 32년 만에 통산 7번째 우승을 기록했습니다.  \n\n특히 코로나19 영향으로 무관중 경기로 진행되었지만, 다저스의 코리 시거, 저스틴 터너 등이 활약한 시리즈였습니다.  \n\n궁금한 점이 있다면 또 물어보세요! 😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 112, 'prompt_tokens': 38, 'total_tokens': 150, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'solar-pro2-250710', 'system_fingerprint': None, 'id': '332fa05b-2c07-4c3e-b51d-11f3ab70b392', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--b1bf974f-0209-42f0-a610-c7a3bfca1447-0', usage_metadata={'input_tokens': 38, 'output_tokens': 112, 'total_tokens': 150, 'input_token_details': {}, 'output_token_details': {}})

In [9]:
ai_message.content

'2020년 월드 시리즈에서는 **로스앤젤레스 다저스**가 우승했습니다!  \nLA 다저스는 탬파베이 레이스와 맞붙어 6차전까지 가는 접전 끝에 4승 2패로 우승을 차지했으며, 32년 만에 통산 7번째 우승을 기록했습니다.  \n\n특히 코로나19 영향으로 무관중 경기로 진행되었지만, 다저스의 코리 시거, 저스틴 터너 등이 활약한 시리즈였습니다.  \n\n궁금한 점이 있다면 또 물어보세요! 😊'